In [187]:
import folium
import geojson
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import plotly.express as px
import plotly.graph_objects as go

from folium import plugins
from folium.plugins import HeatMap
from matplotlib.ticker import FuncFormatter, MultipleLocator
from scipy import stats

In [188]:
df_rent = pd.read_csv('data/mean_rents_all_years.csv')
df_rent['Borough'] = df_rent['Borough'].str.upper()
df_rent.head()

,apartmentType,Borough,2010-01,2010-02,2010-03,2010-04,2010-05,2010-06,2010-07,2010-08,...,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
0,One bedroom,MANHATTAN,2697.98,2624.86,2595.74,2670.08,2648.11,2689.14,2697.67,2732.00,...,4283.44,4364.83,4454.01,4470.85,4516.99,4442.81,4386.88,4430.32,4443.38,4473.64
1,One bedroom,QUEENS,1453.55,1511.83,1550.31,1538.28,1498.15,1567.22,1554.22,1558.67,...,2502.11,2487.07,2542.06,2633.06,2641.88,2623.56,2599.39,2612.71,2617.73,2648.74
2,One bedroom,BROOKLYN,1876.50,1827.00,1838.39,1883.76,1940.65,1931.27,1971.76,1994.36,...,3098.67,3225.65,3240.06,3367.32,3338.79,3391.18,3357.93,3315.09,3245.23,3316.77
3,One bedroom,BRONX,1425.00,1450.00,1450.00,1412.50,1437.50,1400.00,1325.00,1175.00,...,2206.75,2377.05,2340.50,2409.69,2376.83,2409.00,2521.44,2460.56,2383.40,2379.93
4,One bedroom,NEW YORK CITY,2500.00,2500.00,2500.00,2595.00,2600.00,2600.00,2670.00,2695.00,...,3500.00,3650.00,3659.00,3795.00,3850.00,3800.00,3700.00,3700.00,3695.00,3700.00


In [189]:
df_rent[df_rent['Borough']=='STATEN ISLAND']

,apartmentType,Borough,2010-01,2010-02,2010-03,2010-04,2010-05,2010-06,2010-07,2010-08,...,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
5,One bedroom,STATEN ISLAND,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3135.0,3125.0,2850.0,3125.0,2799.0,3125.0,2800.0,3010.0,3310.0,2800.0
11,Two bedrooms,STATEN ISLAND,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3000.0,2867.5,2700.0,3300.0,3300.0,3875.0,3880.0,3525.0,3950.0,3810.0
17,Three plus bedrooms,STATEN ISLAND,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2990.0,NaN,3500.0,3450.0,3100.0,3000.0,3000.0,3250.0,3250.0,3300.0
23,Studio,STATEN ISLAND,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2795.0,2795.0,2550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [272]:
import glob
import pandas as pd

# Function to load and combine all boroughs for a given year
def load_ethnicity_year(year):
    files = glob.glob(f'data/ethnicity_{year}_*.csv')
    dfs = []
    
    for file in files:
        # Extract borough name from filename
        borough = file.split('_')[-1].replace('.csv', '')
        
        # Read CSV
        df = pd.read_csv(file, skiprows=5)
        
        # Keep only the columns you need
        df = df[['Unnamed: 0', 'Number.1', 'Percent.1']].copy()
        
        # Rename columns and add borough
        df.columns = ['ethnicity', 'number', 'percent']
        if borough == 'island':
            df['borough'] = 'STATEN ISLAND'
        else:
            df['borough'] = borough.upper()
        
        dfs.append(df)
    
    # Combine all boroughs
    result = pd.concat(dfs, ignore_index=True)
    return result

# Load both years
ethnicity_2010 = load_ethnicity_year(2010)
ethnicity_2020 = load_ethnicity_year(2020)

print(ethnicity_2010.head(10))
print(ethnicity_2020.head(10))

                          ethnicity     number percent borough
0                  Total Population  1,385,108  100.0%   BRONX
1  Hispanic or Latino (of any race)        NaN     NaN   BRONX
2                           Mexican     71,194    5.1%   BRONX
3                  Central American        NaN     NaN   BRONX
4                       Costa Rican      1,095    0.1%   BRONX
5                        Guatemalan      4,645    0.3%   BRONX
6                          Honduran     17,990    1.3%   BRONX
7                        Nicaraguan      2,342    0.2%   BRONX
8                        Panamanian      2,372    0.2%   BRONX
9                        Salvadoran      5,469    0.4%   BRONX
                          ethnicity     number percent borough
0                  Total Population  1,472,654  100.0%   BRONX
1  Hispanic or Latino (of any race)        NaN     NaN   BRONX
2                           Mexican     76,565    5.2%   BRONX
3                  Central American     39,104    2.7% 

In [273]:
import requests
import pandas as pd

def get_race_data(year):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": "NAME,B02001_001E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E",
        "for": "county:*",
        "in": "state:36"  # New York State
    }
    response = requests.get(url, params=params).json()
    df = pd.DataFrame(response[1:], columns=response[0])
    return df

race_2010 = get_race_data(2010)
race_2020 = get_race_data(2020)


In [274]:
county_to_borough = {
    "Bronx County, New York": "BRONX",
    "Kings County, New York": "BROOKLYN",
    "New York County, New York": "MANHATTAN",
    "Queens County, New York": "QUEENS",
    "Richmond County, New York": "STATEN ISLAND"
}


In [302]:
def clean_race(df):
    df['borough'] = df['NAME'].map(county_to_borough)
    df = df[df['borough'].notna()].copy()

    race_cols = [
        "B02001_001E","B02001_002E","B02001_003E",
        "B02001_004E","B02001_005E","B02001_006E","B02001_007E"
    ]

    for col in race_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

race_2010 = clean_race(race_2010)
race_2020 = clean_race(race_2020)


KeyError: 'B02001_002E'

In [276]:
race_cols = [
    "B02001_001E",  # total
    "B02001_002E",  # white
    "B02001_003E",  # black
    "B02001_004E",  # native
    "B02001_005E",  # asian
    "B02001_006E",  # pacific islander
    "B02001_007E"   # some other race
]

for col in race_cols:
    race_2010[col] = pd.to_numeric(race_2010[col], errors="coerce")
    race_2020[col] = pd.to_numeric(race_2020[col], errors="coerce")


In [277]:
race_2010['borough'].unique()

array([nan, 'BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND'],
      dtype=object)

In [278]:
def clean_ethnicity(df):
    df = df[df['number'].notna()].copy()
    
    # Debug: check what type number is
    print(f"Number dtype before: {df['number'].dtype}")
    print(f"Sample numbers: {df['number'].head()}")
    
    # Convert number column - remove commas first if they're strings
    df['number'] = df['number'].astype(str).str.replace(',', '')
    df['number'] = pd.to_numeric(df['number'], errors='coerce')
    
    print(f"Number dtype after: {df['number'].dtype}")
    print(f"Sample numbers after conversion: {df['number'].head()}")
    
    # Clean ethnicity and borough names
    df['ethnicity'] = df['ethnicity'].str.strip()
    df['borough'] = df['borough'].str.strip().str.upper()
    
    # Get total population for each borough
    total_pop = df[df['ethnicity'] == 'Total Population'].set_index('borough')['number']
    print(f"\nTotal pop by borough:\n{total_pop}")
    
    # Calculate percentage from total population
    df['percent_calc'] = df.apply(
        lambda row: (row['number'] / total_pop.get(row['borough'])) * 100 
        if row['borough'] in total_pop.index and pd.notna(row['number']) 
        else np.nan,
        axis=1
    )
    
    return df

In [279]:
race_2010 = race_2010.rename(columns={
    "B02001_002E": "white_2010",
    "B02001_003E": "black_2010",
    "B02001_004E": "native_2010",
    "B02001_005E": "asian_2010",
    "B02001_006E": "pi_2010",
    "B02001_007E": "other_2010"
})

race_2020 = race_2020.rename(columns={
    "B02001_002E": "white_2020",
    "B02001_003E": "black_2020",
    "B02001_004E": "native_2020",
    "B02001_005E": "asian_2020",
    "B02001_006E": "pi_2020",
    "B02001_007E": "other_2020"
})


In [280]:
list(ethnicity_2010['ethnicity'].unique())

['Total Population',
 'Hispanic or Latino (of any race)',
 'Mexican',
 'Central American',
 'Costa Rican',
 'Guatemalan',
 'Honduran',
 'Nicaraguan',
 'Panamanian',
 'Salvadoran',
 'South American',
 'Argentinean',
 'Bolivian',
 'Chilean',
 'Colombian',
 'Ecuadorian',
 'Paraguayan',
 'Peruvian',
 'Uruguayan',
 'Venezuelan',
 'Caribbean Hispanic',
 'Cuban',
 'Dominican',
 'Puerto Rican',
 'Other Hispanic, Latino, or Spanish',
 'Spaniard',
 'Spanish',
 'Spanish American',
 'Garifuna',
 'White',
 'European',
 'Albanian',
 'Armenian',
 'Austrian',
 'Azerbaijani',
 'Belarusian',
 'Belgian',
 'Bosnian and Herzegovinian',
 'British',
 'Bulgarian',
 'Croatian',
 'Cypriot',
 'Czech',
 'Danish',
 'Dutch',
 'English',
 'Estonian',
 'Finnish',
 'French',
 'Georgian',
 'German',
 'Greek',
 'Hungarian',
 'Irish',
 'Italian',
 'Kosovan',
 'Latvian',
 'Lithuanian',
 'Macedonian',
 'Maltese',
 'Moldovan',
 'Montenegrin',
 'Norwegian',
 'Polish',
 'Portuguese',
 'Romanian',
 'Russian',
 'Scandinavian',


In [281]:
race = race_2010.merge(race_2020, on="borough", suffixes=("_2010", "_2020"))


In [282]:
race

,NAME_2010,B02001_001E_2010,white_2010,black_2010,native_2010,asian_2010,pi_2010,other_2010,state_2010,county_2010,...,NAME_2020,B02001_001E_2020,white_2020,black_2020,native_2020,asian_2020,pi_2020,other_2020,state_2020,county_2020
0,"Albany County, New York",304032,239946,36817,716,13414,185,6504,36,001,...,"Allegany County, New York",46304,44175,757,101,484,26,142,36,003
1,"Albany County, New York",304032,239946,36817,716,13414,185,6504,36,001,...,"Cattaraugus County, New York",76750,70260,1147,2462,541,0,458,36,009
2,"Albany County, New York",304032,239946,36817,716,13414,185,6504,36,001,...,"Chemung County, New York",84115,73504,4992,322,1270,4,490,36,015
3,"Albany County, New York",304032,239946,36817,716,13414,185,6504,36,001,...,"Columbia County, New York",60016,52951,2512,53,1071,6,1216,36,021
4,"Albany County, New York",304032,239946,36817,716,13414,185,6504,36,001,...,"Dutchess County, New York",293524,225543,31432,857,10098,126,10673,36,027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3249,"Yates County, New York",25250,24594,281,78,83,0,37,36,123,...,"Washington County, New York",61304,56773,1829,184,356,23,515,36,115
3250,"Yates County, New York",25250,24594,281,78,83,0,37,36,123,...,"Wayne County, New York",90103,82599,2224,114,766,4,1099,36,117
3251,"Yates County, New York",25250,24594,281,78,83,0,37,36,123,...,"Westchester County, New York",968738,598387,143210,3482,59474,297,113397,36,119
3252,"Yates County, New York",25250,24594,281,78,83,0,37,36,123,...,"Wyoming County, New York",40027,36479,1884,106,182,0,638,36,121


In [283]:
eth10 = clean_ethnicity(ethnicity_2010)
eth20 = clean_ethnicity(ethnicity_2020)


Number dtype before: object
Sample numbers: 0    1,385,108
2       71,194
4        1,095
5        4,645
6       17,990
Name: number, dtype: object
Number dtype after: int64
Sample numbers after conversion: 0    1385108
2      71194
4       1095
5       4645
6      17990
Name: number, dtype: int64

Total pop by borough:
borough
BRONX            1385108
BROOKLYN         2504700
MANHATTAN        1585873
QUEENS           2230722
STATEN ISLAND     468730
Name: number, dtype: int64
Number dtype before: object
Sample numbers: 0    1,472,654
2       76,565
3       39,104
4          989
5        6,809
Name: number, dtype: object
Number dtype after: int64
Sample numbers after conversion: 0    1472654
2      76565
3      39104
4        989
5       6809
Name: number, dtype: int64

Total pop by borough:
borough
BRONX            1472654
BROOKLYN         2736074
MANHATTAN        1694251
QUEENS           2405464
STATEN ISLAND     495747
Name: number, dtype: int64


In [284]:
for group in ["white", "black", "native", "asian", "pi", "other"]:
    race[f"{group}_change_pct"] = (
        (race[f"{group}_2020"] - race[f"{group}_2010"]) / race[f"{group}_2010"]
    )


In [286]:
eth20[eth20['ethnicity']=='Danish']

,ethnicity,number,percent,borough,percent_calc
43,Danish,283,0.0%,BRONX,0.019217
231,Danish,3589,0.1%,BROOKLYN,0.131173
419,Danish,4767,0.3%,MANHATTAN,0.281363
607,Danish,1171,0.0%,QUEENS,0.048681
795,Danish,620,0.1%,STATEN ISLAND,0.125064


In [269]:
race.dtypes


NAME_2010             object
B02001_001E_2010       int64
white_2010             int64
black_2010             int64
native_2010            int64
asian_2010             int64
pi_2010                int64
other_2010             int64
state_2010            object
county_2010           object
borough               object
NAME_2020             object
B02001_001E_2020       int64
white_2020             int64
black_2020             int64
native_2020            int64
asian_2020             int64
pi_2020                int64
other_2020             int64
state_2020            object
county_2020           object
white_change_pct     float64
black_change_pct     float64
native_change_pct    float64
asian_change_pct     float64
pi_change_pct        float64
other_change_pct     float64
dtype: object

In [287]:
group_map = {

    # -------------------------
    # HISPANIC / LATINO
    # -------------------------
    "Hispanic or Latino (of any race)": "Hispanic",
    "Mexican": "Mexican",
    "Central American": "Central American",
    "Costa Rican": "Central American",
    "Guatemalan": "Central American",
    "Honduran": "Central American",
    "Nicaraguan": "Central American",
    "Panamanian": "Central American",
    "Salvadoran": "Central American",

    "South American": "South American",
    "Argentinean": "South American",
    "Bolivian": "South American",
    "Chilean": "South American",
    "Colombian": "South American",
    "Ecuadorian": "South American",
    "Paraguayan": "South American",
    "Peruvian": "South American",
    "Uruguayan": "South American",
    "Venezuelan": "South American",

    "Caribbean Hispanic": "Caribbean Hispanic",
    "Cuban": "Caribbean Hispanic",
    "Dominican": "Caribbean Hispanic",
    "Puerto Rican": "Caribbean Hispanic",

    "Other Hispanic": "Other Hispanic",
    "Spaniard": "Other Hispanic",
    "Spanish": "Other Hispanic",
    "Spanish American": "Other Hispanic",
    "Garifuna": "Other Hispanic",

    # -------------------------
    # WHITE (NON-HISPANIC)
    # -------------------------
    "White": "White",
    "European": "White",
    "Albanian": "White",
    "Armenian": "White",
    "Austrian": "White",
    "Azerbaijani": "White",
    "Belarusian": "White",
    "Belgian": "White",
    "Bosnian and Herzegovinian": "White",
    "British": "White",
    "Bulgarian": "White",
    "Croatian": "White",
    "Cypriot": "White",
    "Czech": "White",
    "Danish": "White",
    "Dutch": "White",
    "English": "White",
    "Estonian": "White",
    "Finnish": "White",
    "French": "White",
    "Georgian": "White",
    "German": "White",
    "Greek": "White",
    "Hungarian": "White",
    "Irish": "White",
    "Italian": "White",
    "Kosovan": "White",
    "Latvian": "White",
    "Lithuanian": "White",
    "Macedonian": "White",
    "Maltese": "White",
    "Moldovan": "White",
    "Montenegrin": "White",
    "Norwegian": "White",
    "Polish": "White",
    "Portuguese": "White",
    "Romanian": "White",
    "Russian": "White",
    "Scandinavian": "White",
    "Scots-Irish": "White",
    "Scottish": "White",
    "Serbian": "White",
    "Slavic": "White",
    "Slovak": "White",
    "Slovenian": "White",
    "Swedish": "White",
    "Swiss": "White",
    "Turkish": "White",
    "Ukrainian": "White",
    "Welsh": "White",

    "Other White": "White",
    "Australian": "White",
    "Canadian": "White",
    "French Canadian": "White",
    "New Zealander": "White",

    # -------------------------
    # MIDDLE EASTERN / NORTH AFRICAN (Middle Eastern)
    # -------------------------
    "Middle Eastern or North African": "Middle Eastern",
    "Algerian": "Middle Eastern",
    "Arab": "Middle Eastern",
    "Egyptian": "Middle Eastern",
    "Iranian": "Middle Eastern",
    "Iraqi": "Middle Eastern",
    "Israeli": "Middle Eastern",
    "Jordanian": "Middle Eastern",
    "Lebanese": "Middle Eastern",
    "Moroccan": "Middle Eastern",
    "Palestinian": "Middle Eastern",
    "Syrian": "Middle Eastern",
    "Tunisian": "Middle Eastern",
    "Yemeni": "Middle Eastern",

    # -------------------------
    # BLACK / AFRICAN AMERICAN
    # -------------------------
    "Black or African American": "Black",
    "African American": "Black",

    "Sub-Saharan African": "Black",
    "Burkinabe": "Black",
    "Cameroonian": "Black",
    "Congolese": "Black",
    "Ethiopian": "Black",
    "Gambian": "Black",
    "Ghanaian": "Black",
    "Guinean": "Black",
    "Ivoirian": "Black",
    "Kenyan": "Black",
    "Liberian": "Black",
    "Malian": "Black",
    "Nigerian (Nigeria)": "Black",
    "Senegalese": "Black",
    "Sierra Leonean": "Black",
    "South African": "Black",
    "Sudanese": "Black",
    "Togolese": "Black",

    "Caribbean": "Black",
    "Antiguan and Barbudan": "Black",
    "Bahamian": "Black",
    "Barbadian": "Black",
    "Dominica Islander": "Black",
    "Grenadian": "Black",
    "Haitian": "Black",
    "Jamaican": "Black",
    "Kittian and Nevisian": "Black",
    "St. Lucian": "Black",
    "Trinidadian and Tobagonian": "Black",
    "U.S. Virgin Islander": "Black",
    "Vincentian": "Black",
    "West Indian": "Black",

    "Other Black or African American": "Black",

    # -------------------------
    # AMERICAN INDIAN / ALASKA NATIVE
    # -------------------------
    "American Indian and Alaska Native": "Native American",
    "Alaska Native": "Native American",
    "American Indian": "Native American",
    "Blackfeet Tribe of the Blackfeet Indian Reservation of Montana": "Native American",
    "Cherokee": "Native American",
    "Central American Indian (all tribes)": "Native American",
    "Mexican Indian (all tribes)": "Native American",
    "Aztec": "Native American",
    "South American Indian (all tribes)": "Native American",
    "Ecuadorian Indian": "Native American",
    "Guyanese South American Indian": "Native American",
    "Inca": "Native American",
    "Caribbean Indian (all tribes)": "Native American",
    "Taino": "Native American",
    "Mesoamerican Indian (all tribes)": "Native American",
    "Maya": "Native American",

    # -------------------------
    # ASIAN
    # -------------------------
    "Asian": "Asian",
    "East Asian": "Asian",
    "Chinese,  except Taiwanese": "Asian",
    "Japanese": "Asian",
    "Korean": "Asian",
    "Taiwanese": "Asian",

    "Central Asian": "Asian",
    "Afghan": "Asian",
    "Kazakh": "Asian",
    "Kyrgyz": "Asian",
    "Tajik": "Asian",
    "Uzbek": "Asian",

    "South Asian": "Asian",
    "Asian Indian": "Asian",
    "Bangladeshi": "Asian",
    "Nepalese": "Asian",
    "Pakistani": "Asian",
    "Sikh": "Asian",
    "Sri Lankan": "Asian",

    "Southeast Asian": "Asian",
    "Burmese": "Asian",
    "Cambodian": "Asian",
    "Filipino": "Asian",
    "Indonesian": "Asian",
    "Malaysian": "Asian",
    "Singaporean": "Asian",
    "Thai": "Asian",
    "Vietnamese": "Asian",

    "Other Asian": "Asian",

    # -------------------------
    # PACIFIC ISLANDER
    # -------------------------
    "Native Hawaiian and Other Pacific Islander": "Pacific Islander",
    "Polynesian": "Pacific Islander",
    "Native Hawaiian": "Pacific Islander",
    "Samoan": "Pacific Islander",
    "Micronesian": "Pacific Islander",
    "Chamorro": "Pacific Islander",

    # -------------------------
    # SOME OTHER RACE
    # -------------------------
    "Some Other Race": "Some Other Race",
    "Belizean": "Some Other Race",
    "Brazilian": "Some Other Race",
    "Guyanese": "Some Other Race",
}


In [288]:
def collapse_groups(df):
    df = df[df['ethnicity'].isin(group_map.keys())].copy()
    df['group'] = df['ethnicity'].map(group_map)
    grouped = df.groupby(['borough', 'group'])['number'].sum().reset_index()
    return grouped

eth10g = collapse_groups(eth10)
eth20g = collapse_groups(eth20)


In [289]:
eth10g

,borough,group,number
0,BRONX,Asian,47910
1,BRONX,Caribbean Hispanic,548693
2,BRONX,Central American,33913
3,BRONX,Mexican,71194
4,BRONX,South American,35094
5,BROOKLYN,Asian,99356
6,BROOKLYN,Caribbean Hispanic,270873
7,BROOKLYN,Central American,45632
8,BROOKLYN,Mexican,94585
9,BROOKLYN,South American,48497


In [290]:
sorted(eth10['ethnicity'].unique())


['Argentinean',
 'Asian Indian',
 'Bangladeshi',
 'Bolivian',
 'Cambodian',
 'Chilean',
 'Chinese, except Taiwanese',
 'Colombian',
 'Costa Rican',
 'Cuban',
 'Dominican',
 'Ecuadorian',
 'Filipino',
 'Guatemalan',
 'Honduran',
 'Indonesian',
 'Japanese',
 'Korean',
 'Malaysian',
 'Mexican',
 'Nicaraguan',
 'Pakistani',
 'Panamanian',
 'Paraguayan',
 'Peruvian',
 'Puerto Rican',
 'Salvadoran',
 'Sri Lankan',
 'Taiwanese',
 'Thai',
 'Total Population',
 'Uruguayan',
 'Venezuelan',
 'Vietnamese']

In [296]:
def collapse_groups(df):
    df = df[df['ethnicity'].isin(group_map.keys())].copy()
    df['group'] = df['ethnicity'].map(group_map)
    grouped = df.groupby(['borough', 'group'])['percent_calc'].sum().reset_index()
    grouped = grouped.rename(columns={'percent_calc': 'pct'})
    return grouped


eth10g = collapse_groups(eth10)
eth20g = collapse_groups(eth20)


In [293]:
eth10g

,borough,group,number
0,BRONX,Asian,47910
1,BRONX,Caribbean Hispanic,548693
2,BRONX,Central American,33913
3,BRONX,Mexican,71194
4,BRONX,South American,35094
5,BROOKLYN,Asian,99356
6,BROOKLYN,Caribbean Hispanic,270873
7,BROOKLYN,Central American,45632
8,BROOKLYN,Mexican,94585
9,BROOKLYN,South American,48497


In [297]:
eth_change = eth10g.merge(
    eth20g,
    on=['borough', 'group'],
    suffixes=('_2010', '_2020')
)


eth_change['change_pct'] = (
    (eth_change['pct_2020'] - eth_change['pct_2010']) / eth_change['pct_2010']
)


In [298]:
rent_cols_2010 = [c for c in df_rent.columns if c.startswith("2010")]
rent_cols_2020 = [c for c in df_rent.columns if c.startswith("2020")]

df_rent['rent_2010_mean'] = df_rent[rent_cols_2010].mean(axis=1)
df_rent['rent_2020_mean'] = df_rent[rent_cols_2020].mean(axis=1)

df_rent['rent_change_pct'] = (
    (df_rent['rent_2020_mean'] - df_rent['rent_2010_mean'])
    / df_rent['rent_2010_mean']
)


In [299]:
rent_change = df_rent[['Borough', 'apartmentType', 'rent_change_pct']]


In [ ]:
df_merged = eth_change.merge(rent_change, on='borough')
df_merged = df_merged.merge(race, on='borough')


KeyError: 'borough'

In [204]:
df_merged = df_merged.drop(columns=['Borough'])


In [205]:
corr = (
    df_merged.groupby('group')[['change_pct', 'rent_change_pct']]
    .corr()
    .iloc[0::2, -1]
)

print(corr)


group                         
Asian               change_pct   -0.359163
Black               change_pct         NaN
Caribbean Hispanic  change_pct    0.561099
Central American    change_pct    0.510127
Hispanic            change_pct         NaN
Mexican             change_pct    0.488607
Middle Eastern      change_pct         NaN
Native American     change_pct         NaN
Other Hispanic      change_pct         NaN
Pacific Islander    change_pct         NaN
Some Other Race     change_pct         NaN
South American      change_pct   -0.450735
White               change_pct         NaN
Name: rent_change_pct, dtype: float64


In [206]:
import plotly.express as px

fig = px.scatter(
    df_merged,
    x="change_pct",
    y="rent_change_pct",
    color="group",
    hover_name="borough",
    hover_data={
        "change_pct": ":.2%",
        "rent_change_pct": ":.2%",
        "group": True,
        "borough": True
    },
    labels={
        "change_pct": "Ethnicity % Change (2010 → 2020)",
        "rent_change_pct": "Rent % Change (2010 → 2020)",
        "group": "Ethnicity Group"
    },
    title="Correlation Between Ethnicity Change and Rent Change (2010–2020)"
)

fig.update_traces(marker=dict(size=14, opacity=0.8))

fig.show()


In [207]:
def safe_regression(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)

    # Remove NaN / inf
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    # Need at least 2 points
    if len(x) < 2:
        return None, None

    # Need variance in x
    if np.allclose(x, x[0]):
        return None, None

    # Compute slope and intercept manually
    x_mean = x.mean()
    y_mean = y.mean()

    m = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)
    b = y_mean - m * x_mean

    return m, b


In [208]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Get unique ethnicity groups
groups = df_merged['group'].unique()
n_groups = len(groups)

# Create subplot grid (auto layout: 2 columns)
cols = 2
rows = int(np.ceil(n_groups / cols))

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=groups,
    horizontal_spacing=0.12,
    vertical_spacing=0.12
)

row = 1
col = 1

for group in groups:
    sub = df_merged[df_merged['group'] == group]

    # Scatter points
    fig.add_trace(
        go.Scatter(
            x=sub['change_pct'],
            y=sub['rent_change_pct'],
            mode='markers',
            name=group,
            marker=dict(size=10),
            text=sub['borough'],
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "Ethnicity change: %{x:.2%}<br>" +
                "Rent change: %{y:.2%}<br>"
            )
        ),
        row=row, col=col
    )

    # Regression line
    x = sub['change_pct']
    y = sub['rent_change_pct']

    m, b = safe_regression(x, y)

    if m is not None:
        x_line = np.linspace(x.min(), x.max(), 50)
        y_line = m * x_line + b

        fig.add_trace(
            go.Scatter(
                x=x_line,
                y=y_line,
                mode='lines',
                line=dict(color='black', width=2),
                showlegend=False
            ),
            row=row, col=col
        )
    else:
        print(f"Skipping regression for {group} (insufficient data or variance)")

    # Move to next subplot
    col += 1
    if col > cols:
        col = 1
        row += 1


# Layout
fig.update_layout(
    height=300 * rows,
    width=900,
    title="Ethnicity Change vs Rent Change (2010–2020) — Per Ethnicity Group",
    showlegend=False
)

fig.update_xaxes(title_text="Ethnicity % Change")
fig.update_yaxes(title_text="Rent % Change")

fig.show()


Skipping regression for Black (insufficient data or variance)
Skipping regression for Hispanic (insufficient data or variance)
Skipping regression for Middle Eastern (insufficient data or variance)
Skipping regression for Native American (insufficient data or variance)
Skipping regression for Other Hispanic (insufficient data or variance)
Skipping regression for Pacific Islander (insufficient data or variance)
Skipping regression for Some Other Race (insufficient data or variance)
Skipping regression for White (insufficient data or variance)
